In [1]:
workspace_id = "2f59ccc2-e9d6-44b5-8c0c-1bdbf1fe148a"
bronze_lakehouse_id = "33b93b7e-0899-43c6-81a8-654bc740aa32"

StatementMeta(, a15034db-966d-40d7-b6e7-76e1c4c93ea5, 3, Finished, Available, Finished, False)

In [2]:
import pandas as pd
from pyspark.sql import functions as F

# --- CONFIGURATION & PARAMETERS ---
BRONZE_BASE_PATH = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Tables/dbo"
ZONES_CSV_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# 1. Fetch official NYC TLC Taxi Zone Lookup CSV into Bronze
df_zones_pd = pd.read_csv(ZONES_CSV_URL)
df_zones_bronze = spark.createDataFrame(df_zones_pd)

df_zones_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    f"{BRONZE_BASE_PATH}/nyc_taxi_zones_bronze"
)

# 2. Clean and standardize columns for Silver layer
df_zones_silver = (
    df_zones_bronze
    .select(
        F.col("LocationID").cast("int").alias("location_id"),
        F.trim(F.col("Borough")).alias("borough"),
        F.trim(F.col("Zone")).alias("zone_name"),
        F.trim(F.col("service_zone")).alias("service_zone")
    )
    .filter(F.col("location_id").isNotNull())
    .dropDuplicates(["location_id"])
)

df_zones_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nyc_taxi_zones_silver")

print(f"Successfully saved nyc_taxi_zones_silver. Record count: {df_zones_silver.count()}")
display(df_zones_silver.limit(10))

StatementMeta(, a15034db-966d-40d7-b6e7-76e1c4c93ea5, 4, Finished, Available, Finished, False)

Successfully saved nyc_taxi_zones_silver. Record count: 265


SynapseWidget(Synapse.DataFrame, 2c6477c7-7970-490f-8db3-938ee43b327b)